In [3]:
!pip install dagshub mlflow optuna -q

In [4]:
import os
import gc
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

import mlflow

import dagshub
dagshub.init (
    repo_owner="sophyrise",
    repo_name='ieee-cis-fraud-detection',
    mlflow=True
)

mlflow.set_experiment("Random Forest")
print("✅ MLflow tracking URI:", mlflow.get_tracking_uri())

Initialized MLflow to track repo "sophyrise/ieee-cis-fraud-detection"

Repository sophyrise/ieee-cis-fraud-detection initialized!

✅ MLflow tracking URI: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow


# **Cleaning**

In [ ]:
with mlflow.start_run(run_name="RandomForest_Cleaning"):
    DATA_DIR = "/kaggle/input/competitions/ieee-fraud-detection"
    TXN_MISSING_THRESHOLD = 0.80
    ID_MISSING_THRESHOLD = 0.95
    NEAR_CONSTANT_THRESHOLD = 0.99

    train_txn = pd.read_csv(f"{DATA_DIR}/train_transaction.csv")
    train_id  = pd.read_csv(f"{DATA_DIR}/train_identity.csv")
    test_txn  = pd.read_csv(f"{DATA_DIR}/test_transaction.csv")
    test_id   = pd.read_csv(f"{DATA_DIR}/test_identity.csv")

    test_id.columns = test_id.columns.str.replace("-", "_", regex=False)

    train = train_txn.merge(train_id, on="TransactionID", how="left")
    test  = test_txn.merge(test_id, on="TransactionID", how="left")

    del train_txn, train_id, test_txn, test_id
    gc.collect()

    y_train_rf = train["isFraud"].astype(np.int8)
    X_train_rf = train.drop(columns=["isFraud", "TransactionID"])
    X_test_rf  = test.drop(columns=["TransactionID"])

    del train, test
    gc.collect()

    id_like_cols = [c for c in X_train_rf.columns if c.startswith("id_") or c in ["DeviceType", "DeviceInfo"]]
    txn_like_cols = [c for c in X_train_rf.columns if c not in id_like_cols]

    missing_ratio = X_train_rf.isnull().mean()

    drop_txn = [c for c in txn_like_cols if missing_ratio[c] > TXN_MISSING_THRESHOLD]
    drop_id  = [c for c in id_like_cols if missing_ratio[c] > ID_MISSING_THRESHOLD]
    drop_missing = drop_txn + drop_id

    X_train_rf = X_train_rf.drop(columns=drop_missing)
    X_test_rf  = X_test_rf.drop(columns=[c for c in drop_missing if c in X_test_rf.columns])

    near_constant_cols = []
    for col in X_train_rf.columns:
        top_freq = X_train_rf[col].value_counts(dropna=False, normalize=True).iloc[0]
        if top_freq > NEAR_CONSTANT_THRESHOLD:
            near_constant_cols.append(col)

    X_train_rf = X_train_rf.drop(columns=near_constant_cols)
    X_test_rf  = X_test_rf.drop(columns=[c for c in near_constant_cols if c in X_test_rf.columns])

    for col in X_train_rf.columns:
        if col not in X_test_rf.columns:
            X_test_rf[col] = np.nan
    X_test_rf = X_test_rf[X_train_rf.columns]

    mlflow.log_param("txn_missing_threshold", TXN_MISSING_THRESHOLD)
    mlflow.log_param("id_missing_threshold", ID_MISSING_THRESHOLD)
    mlflow.log_param("near_constant_threshold", NEAR_CONSTANT_THRESHOLD)

    mlflow.log_metric("train_rows", int(X_train_rf.shape[0]))
    mlflow.log_metric("test_rows", int(X_test_rf.shape[0]))
    mlflow.log_metric("features_after_cleaning", int(X_train_rf.shape[1]))
    mlflow.log_metric("fraud_rate", float(y_train_rf.mean()))

    print("X_train_clean_rf:", X_train_rf.shape)
    print("X_test_clean_rf: ", X_test_rf.shape)

    X_train_clean_rf = X_train_rf
    X_test_clean_rf = X_test_rf
    y_train_clean_rf = y_train_rf

# **Feature Engineering**

In [ ]:
with mlflow.start_run(run_name="RandomForest_FeatureEngineering"):
    from sklearn.impute import SimpleImputer

    X_train = X_train_clean_rf.copy()
    X_test = X_test_clean_rf.copy()
    y_train = y_train_clean_rf.copy()

    # simple strong numeric signals
    X_train["TransactionAmt_log"] = np.log1p(X_train["TransactionAmt"].clip(lower=0))
    X_test["TransactionAmt_log"] = np.log1p(X_test["TransactionAmt"].clip(lower=0))

    X_train["hour_sin"] = np.sin(2 * np.pi * ((X_train["TransactionDT"] // 3600) % 24) / 24)
    X_train["hour_cos"] = np.cos(2 * np.pi * ((X_train["TransactionDT"] // 3600) % 24) / 24)
    X_test["hour_sin"] = np.sin(2 * np.pi * ((X_test["TransactionDT"] // 3600) % 24) / 24)
    X_test["hour_cos"] = np.cos(2 * np.pi * ((X_test["TransactionDT"] // 3600) % 24) / 24)

    X_train = X_train.drop(columns=["TransactionDT"], errors="ignore")
    X_test = X_test.drop(columns=["TransactionDT"], errors="ignore")

    cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
    num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

    num_imp = SimpleImputer(strategy="median")
    X_train[num_cols] = num_imp.fit_transform(X_train[num_cols])
    X_test[num_cols] = num_imp.transform(X_test[num_cols])

    # ordinal encoding from train categories
    for c in cat_cols:
        uniq = pd.Series(X_train[c].astype(str).unique())
        mapping = {v: i for i, v in enumerate(uniq)}
        X_train[c] = X_train[c].astype(str).map(mapping).fillna(-1).astype(np.int32)
        X_test[c] = X_test[c].astype(str).map(mapping).fillna(-1).astype(np.int32)

    X_test = X_test.reindex(columns=X_train.columns, fill_value=-1)

    mlflow.log_param("numeric_imputer", "median")
    mlflow.log_param("cat_encoding", "ordinal_train_mapping")
    mlflow.log_metric("features_after_fe", int(X_train.shape[1]))
    mlflow.log_metric("cat_cols_encoded", int(len(cat_cols)))

    print("X_train_fe_rf:", X_train.shape)
    print("X_test_fe_rf: ", X_test.shape)

    X_train_fe_rf = X_train
    X_test_fe_rf = X_test
    y_train_fe_rf = y_train

# **Feature Selection**

In [ ]:
with mlflow.start_run(run_name="RandomForest_FeatureSelection"):
    X_train = X_train_fe_rf.copy()
    X_test = X_test_fe_rf.copy()

    X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
    X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

    nu = X_train.nunique(dropna=False)
    const_cols = nu[nu <= 1].index.tolist()
    X_train = X_train.drop(columns=const_cols, errors="ignore")
    X_test = X_test.drop(columns=const_cols, errors="ignore")

    CORR_THRESHOLD = 0.98
    num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

    sample_n = min(120_000, len(X_train))
    idx = np.random.RandomState(42).choice(len(X_train), size=sample_n, replace=False)

    corr = X_train.iloc[idx][num_cols].corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
    drop_corr = [c for c in upper.columns if (upper[c] > CORR_THRESHOLD).any()]

    X_train = X_train.drop(columns=drop_corr, errors="ignore")
    X_test = X_test.drop(columns=drop_corr, errors="ignore")

    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

    mlflow.log_param("corr_threshold", CORR_THRESHOLD)
    mlflow.log_metric("dropped_const", len(const_cols))
    mlflow.log_metric("dropped_corr", len(drop_corr))
    mlflow.log_metric("features_after_fs", int(X_train.shape[1]))

    print("X_train_fs_rf:", X_train.shape)
    print("X_test_fs_rf: ", X_test.shape)

    X_train_final_rf = X_train
    X_test_final_rf = X_test

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
import pickle, os
import warnings
warnings.filterwarnings('ignore')

CHECKPOINT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

with open(f"{CHECKPOINT_DIR}/X_train_final_rf.pkl", "wb") as f:
    pickle.dump(X_train_final_rf, f)
with open(f"{CHECKPOINT_DIR}/X_test_final_rf.pkl", "wb") as f:
    pickle.dump(X_test_final_rf, f)
with open(f"{CHECKPOINT_DIR}/y_train_fe_rf.pkl", "wb") as f:
    pickle.dump(y_train_fe_rf, f)

print("✅ Checkpoint saved to", CHECKPOINT_DIR)

# Training

In [5]:
import pickle, os

CHECKPOINT_DIR = "/kaggle/working/checkpoints"

with open(f"{CHECKPOINT_DIR}/X_train_final_rf.pkl", "rb") as f:
    X_train_final_rf = pickle.load(f)
with open(f"{CHECKPOINT_DIR}/X_test_final_rf.pkl", "rb") as f:
    X_test_final_rf = pickle.load(f)
with open(f"{CHECKPOINT_DIR}/y_train_fe_rf.pkl", "rb") as f:
    y_train_fe_rf = pickle.load(f)

print("✅ Loaded from checkpoint")
print(f"X_train: {X_train_final_rf.shape} | X_test: {X_test_final_rf.shape}")

✅ Loaded from checkpoint
X_train: (590540, 306) | X_test: (506691, 306)


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
import pickle, os
import warnings
warnings.filterwarnings('ignore')

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_final_rf, y_train_fe_rf,
    test_size=0.2, random_state=42, stratify=y_train_fe_rf
)
print(f"Train: {X_tr.shape} | Val: {X_val.shape}")

Train: (472432, 306) | Val: (118108, 306)


In [7]:
with mlflow.start_run(run_name="RF_Baseline"):
    mlflow.log_param("n_estimators",  100)
    mlflow.log_param("max_depth",     5)
    mlflow.log_param("class_weight",  None)
    mlflow.log_param("note",          "shallow trees, no class balancing — expect underfitting")

    clf = RandomForestClassifier(n_estimators=100, max_depth=5,
                                 random_state=42, n_jobs=-1)
    clf.fit(X_tr, y_tr)

    train_auc = roc_auc_score(y_tr,  clf.predict_proba(X_tr)[:, 1])
    val_auc   = roc_auc_score(y_val, clf.predict_proba(X_val)[:, 1])
    gap = train_auc - val_auc

    mlflow.log_metric("train_auc",   round(train_auc, 5))
    mlflow.log_metric("val_auc",     round(val_auc,   5))
    mlflow.log_metric("overfit_gap", round(gap, 5))

    print(f"[Baseline] Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {gap:.4f}")
    print(f"  -> UNDERFIT as expected: depth=5, no class balancing")

[Baseline] Train: 0.8415 | Val: 0.8411 | Gap: 0.0004
  -> UNDERFIT as expected: depth=5, no class balancing
🏃 View run RF_Baseline at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/3/runs/e783ad6b52be4bc1ab11f657c9509df9
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/3


In [8]:
# RF has balanced_subsample which rebalances per tree
# better than global balanced for large imbalanced datasets like fraud
cw_results = []
for cw in [None, "balanced_subsample"]:
    label = str(cw)
    with mlflow.start_run(run_name=f"RF_classweight_{label}"):
        mlflow.log_param("n_estimators",  100)
        mlflow.log_param("max_depth",     10)
        mlflow.log_param("class_weight",  label)

        clf = RandomForestClassifier(n_estimators=100, max_depth=10,
                                     class_weight=cw,
                                     random_state=42, n_jobs=-1)
        clf.fit(X_tr, y_tr)

        train_auc = roc_auc_score(y_tr,  clf.predict_proba(X_tr)[:, 1])
        val_auc   = roc_auc_score(y_val, clf.predict_proba(X_val)[:, 1])
        gap = train_auc - val_auc

        mlflow.log_metric("train_auc",   round(train_auc, 5))
        mlflow.log_metric("val_auc",     round(val_auc,   5))
        mlflow.log_metric("overfit_gap", round(gap, 5))

        cw_results.append({"class_weight": label, "val_auc": val_auc})
        print(f"  class_weight={label:<25} | Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {gap:.4f}")

cw_df = pd.DataFrame(cw_results).set_index("class_weight")
best_cw_label = cw_df["val_auc"].idxmax()
best_cw = None if best_cw_label == "None" else best_cw_label
print(f"\n-> Best class_weight: {best_cw_label}")

  class_weight=None                      | Train: 0.8762 | Val: 0.8698 | Gap: 0.0064
🏃 View run RF_classweight_None at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/3/runs/0c384aafca1f4424a1ced17447aa665d
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/3
  class_weight=balanced_subsample        | Train: 0.8970 | Val: 0.8845 | Gap: 0.0125
🏃 View run RF_classweight_balanced_subsample at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/3/runs/67179420765b489c860587a5585a36f7
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/3

-> Best class_weight: balanced_subsample


In [9]:
depth_results = []
for depth in [5, 10, 20, None]:
    label = str(depth)
    with mlflow.start_run(run_name=f"RF_depth_{label}"):
        mlflow.log_param("n_estimators",  100)
        mlflow.log_param("max_depth",     label)
        mlflow.log_param("class_weight",  best_cw_label)

        clf = RandomForestClassifier(n_estimators=100, max_depth=depth,
                                     class_weight=best_cw,
                                     random_state=42, n_jobs=-1)
        clf.fit(X_tr, y_tr)

        train_auc = roc_auc_score(y_tr,  clf.predict_proba(X_tr)[:, 1])
        val_auc   = roc_auc_score(y_val, clf.predict_proba(X_val)[:, 1])
        gap = train_auc - val_auc

        mlflow.log_metric("train_auc",   round(train_auc, 5))
        mlflow.log_metric("val_auc",     round(val_auc,   5))
        mlflow.log_metric("overfit_gap", round(gap, 5))

        if gap < 0.01:
            diagnosis = "UNDERFIT — trees too shallow"
        elif gap > 0.05:
            diagnosis = "OVERFIT — trees too deep, memorising"
        else:
            diagnosis = "OK"

        depth_results.append({"max_depth": label, "train_auc": train_auc,
                               "val_auc": val_auc, "gap": gap})
        print(f"  depth={label:<5} | Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {gap:.4f} — {diagnosis}")

depth_df = pd.DataFrame(depth_results).set_index("max_depth")
best_depth_label = depth_df["val_auc"].idxmax()
best_depth_val   = None if best_depth_label == "None" else int(best_depth_label)
print(f"\n-> Best depth: {best_depth_label}")
print(depth_df.to_string())

  depth=5     | Train: 0.8459 | Val: 0.8445 | Gap: 0.0014 — UNDERFIT — trees too shallow
🏃 View run RF_depth_5 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/3/runs/6657b7dba892436eafd49534cdfc3556
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/3
  depth=10    | Train: 0.8970 | Val: 0.8845 | Gap: 0.0125 — OK
🏃 View run RF_depth_10 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/3/runs/99544c6b1be041cbb87471ca86d75634
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/3
  depth=20    | Train: 0.9849 | Val: 0.9246 | Gap: 0.0603 — OVERFIT — trees too deep, memorising
🏃 View run RF_depth_20 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/3/runs/817c73138c0642e380a306fc71d8c599
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/3
  depth=None  | Train: 

In [10]:
# more trees = better generalization up to a point
# RF doesn't overfit with more trees but has diminishing returns
nest_results = []
for n_est in [100, 200, 300]:
    with mlflow.start_run(run_name=f"RF_nestimators_{n_est}"):
        mlflow.log_param("n_estimators",  n_est)
        mlflow.log_param("max_depth",     best_depth_label)
        mlflow.log_param("class_weight",  best_cw_label)

        clf = RandomForestClassifier(n_estimators=n_est, max_depth=best_depth_val,
                                     class_weight=best_cw,
                                     random_state=42, n_jobs=-1)
        clf.fit(X_tr, y_tr)

        train_auc = roc_auc_score(y_tr,  clf.predict_proba(X_tr)[:, 1])
        val_auc   = roc_auc_score(y_val, clf.predict_proba(X_val)[:, 1])
        gap = train_auc - val_auc

        mlflow.log_metric("train_auc",   round(train_auc, 5))
        mlflow.log_metric("val_auc",     round(val_auc,   5))
        mlflow.log_metric("overfit_gap", round(gap, 5))

        nest_results.append({"n_estimators": n_est, "train_auc": train_auc,
                              "val_auc": val_auc, "gap": gap})
        print(f"  n_estimators={n_est:<4} | Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {gap:.4f}")

nest_df = pd.DataFrame(nest_results).set_index("n_estimators")
best_n_est = int(nest_df["val_auc"].idxmax())
print(f"\n-> Best n_estimators: {best_n_est}")
print(nest_df.to_string())

  n_estimators=100  | Train: 1.0000 | Val: 0.9415 | Gap: 0.0585
🏃 View run RF_nestimators_100 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/3/runs/0dcb2757b6fb4b0a87100ba5d31ea925
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/3
  n_estimators=200  | Train: 1.0000 | Val: 0.9461 | Gap: 0.0539
🏃 View run RF_nestimators_200 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/3/runs/e493dd6faa80409c975f86cbd4091aa3
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/3
  n_estimators=300  | Train: 1.0000 | Val: 0.9475 | Gap: 0.0525
🏃 View run RF_nestimators_300 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/3/runs/691ef0b33d9e4bac9aa1834c9c94bd3f
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/3

-> Best n_estimators: 300
              train_auc   val_a

In [11]:
# min_samples_leaf forces leaves to have at least N samples
# effectively regularizes overfitting in deep trees
reg_results = []
for min_samples in [1, 10, 50, 100]:
    with mlflow.start_run(run_name=f"RF_minsamples_{min_samples}"):
        mlflow.log_param("n_estimators",     best_n_est)
        mlflow.log_param("max_depth",        best_depth_label)
        mlflow.log_param("class_weight",     best_cw_label)
        mlflow.log_param("min_samples_leaf", min_samples)

        clf = RandomForestClassifier(n_estimators=best_n_est, max_depth=best_depth_val,
                                     class_weight=best_cw,
                                     min_samples_leaf=min_samples,
                                     random_state=42, n_jobs=-1)
        clf.fit(X_tr, y_tr)

        train_auc = roc_auc_score(y_tr,  clf.predict_proba(X_tr)[:, 1])
        val_auc   = roc_auc_score(y_val, clf.predict_proba(X_val)[:, 1])
        gap = train_auc - val_auc

        mlflow.log_metric("train_auc",   round(train_auc, 5))
        mlflow.log_metric("val_auc",     round(val_auc,   5))
        mlflow.log_metric("overfit_gap", round(gap, 5))

        reg_results.append({"min_samples_leaf": min_samples,
                             "train_auc": train_auc, "val_auc": val_auc, "gap": gap})
        print(f"  min_samples_leaf={min_samples:<5} | Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {gap:.4f}")

reg_df = pd.DataFrame(reg_results).set_index("min_samples_leaf")
best_min_samples = int(reg_df["val_auc"].idxmax())
print(f"\n-> Best min_samples_leaf: {best_min_samples}")
print(reg_df.to_string())

  min_samples_leaf=1     | Train: 1.0000 | Val: 0.9475 | Gap: 0.0525
🏃 View run RF_minsamples_1 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/3/runs/550f5a523f354ce496e97569aff28f59
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/3
  min_samples_leaf=10    | Train: 0.9955 | Val: 0.9405 | Gap: 0.0550
🏃 View run RF_minsamples_10 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/3/runs/f5378fe7d5ff475c83bf5cfa8a4f9a3b
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/3
  min_samples_leaf=50    | Train: 0.9661 | Val: 0.9216 | Gap: 0.0445
🏃 View run RF_minsamples_50 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/3/runs/be9e8b3559584aef9d7c8c7d2b2f9ed1
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/3
  min_samples_leaf=100   | Train: 0.9450 | Val: 0.

In [12]:
with mlflow.start_run(run_name="RF_CrossValidation_5fold"):
    mlflow.log_param("n_estimators",     best_n_est)
    mlflow.log_param("max_depth",        best_depth_label)
    mlflow.log_param("class_weight",     best_cw_label)
    mlflow.log_param("min_samples_leaf", best_min_samples)
    mlflow.log_param("cv_folds",         5)

    clf_cv = RandomForestClassifier(n_estimators=best_n_est, max_depth=best_depth_val,
                                    class_weight=best_cw,
                                    min_samples_leaf=best_min_samples,
                                    random_state=42, n_jobs=-1)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(clf_cv, X_train_final_rf, y_train_fe_rf,
                                cv=cv, scoring="roc_auc", n_jobs=-1)

    for i, score in enumerate(cv_scores):
        mlflow.log_metric("fold_auc", round(score, 5), step=i)

    mlflow.log_metric("cv_mean_auc", round(cv_scores.mean(), 5))
    mlflow.log_metric("cv_std_auc",  round(cv_scores.std(),  5))

    print(f"CV folds: {[round(s, 4) for s in cv_scores]}")
    print(f"Mean: {cv_scores.mean():.4f} | Std: {cv_scores.std():.4f}")
    print(f"  -> {'STABLE' if cv_scores.std() < 0.005 else 'HIGH VARIANCE'}")

CV folds: [np.float64(0.9448), np.float64(0.9492), np.float64(0.943), np.float64(0.9459), np.float64(0.9474)]
Mean: 0.9461 | Std: 0.0021
  -> STABLE
🏃 View run RF_CrossValidation_5fold at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/3/runs/ecf989ca489a4e64be8703395b219f88
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/3


In [13]:
with mlflow.start_run(run_name="RF_Final_Pipeline") as run:
    mlflow.log_param("n_estimators",     best_n_est)
    mlflow.log_param("max_depth",        best_depth_label)
    mlflow.log_param("class_weight",     best_cw_label)
    mlflow.log_param("min_samples_leaf", best_min_samples)
    mlflow.log_param("note",             "best params from all sweeps, honest val holdout")

    final_pipe = Pipeline([
        ("clf", RandomForestClassifier(
            n_estimators=best_n_est,
            max_depth=best_depth_val,
            class_weight=best_cw,
            min_samples_leaf=best_min_samples,
            random_state=42,
            n_jobs=-1
        ))
    ])

    final_pipe.fit(X_tr, y_tr)

    train_auc = roc_auc_score(y_tr,  final_pipe.predict_proba(X_tr)[:, 1])
    val_auc   = roc_auc_score(y_val, final_pipe.predict_proba(X_val)[:, 1])
    gap = train_auc - val_auc

    mlflow.log_metric("train_auc",   round(train_auc, 5))
    mlflow.log_metric("val_auc",     round(val_auc,   5))
    mlflow.log_metric("overfit_gap", round(gap, 5))

    mlflow.sklearn.log_model(
        sk_model=final_pipe,
        artifact_path="random_forest_pipeline",
        registered_model_name="RandomForest_FraudDetection"
    )

    pkl_path = "/kaggle/working/random_forest_final_pipeline.pkl"
    with open(pkl_path, "wb") as f:
        pickle.dump(final_pipe, f)
    mlflow.log_artifact(pkl_path)

    print(f"[Final] Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {gap:.4f}")
    print(f"  -> {'OVERFIT' if gap > 0.05 else 'OK'}")
    print(f"Run ID: {run.info.run_id}")
    print("✅ Registered as RandomForest_FraudDetection")

2026/05/03 11:47:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 11:47:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'RandomForest_FraudDetection' already exists. Creating a new version of this model...
2026/05/03 11:49:30 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: RandomForest_FraudDetection, version 4
Created version '4' of model 'RandomForest_FraudDetection'.


[Final] Train: 1.0000 | Val: 0.9475 | Gap: 0.0525
  -> OVERFIT
Run ID: f3fb388ecbc44f60b707005e79aaba82
✅ Registered as RandomForest_FraudDetection
🏃 View run RF_Final_Pipeline at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/3/runs/f3fb388ecbc44f60b707005e79aaba82
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/3


# Graph and Pictures

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

depth_data = {
    "depth=5":    {"train_auc": 0.8459, "val_auc": 0.8445},
    "depth=10":   {"train_auc": 0.8970, "val_auc": 0.8845},
    "depth=20":   {"train_auc": 0.9849, "val_auc": 0.9246},
    "depth=None": {"train_auc": 1.0000, "val_auc": 0.9415},
}

df_depth = pd.DataFrame(depth_data).T
df_depth["gap"] = df_depth["train_auc"] - df_depth["val_auc"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = range(len(df_depth))
axes[0].plot(list(x), df_depth["train_auc"], marker="o", label="Train AUC", color="#4C72B0")
axes[0].plot(list(x), df_depth["val_auc"],   marker="o", label="Val AUC",   color="#DD8452")
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(df_depth.index, rotation=20, ha="right")
axes[0].set_title("Train vs Val AUC by Depth\n(underfitting → overfitting)")
axes[0].set_ylabel("AUC")
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].annotate("underfit", xy=(0, 0.845), fontsize=8, color="gray")
axes[0].annotate("overfit zone", xy=(2.3, 0.96), fontsize=8, color="gray")

axes[1].bar(df_depth.index, df_depth["gap"],
            color=["#55A868" if g < 0.05 else "#C44E52" for g in df_depth["gap"]])
axes[1].set_xticklabels(df_depth.index, rotation=20, ha="right")
axes[1].set_title("Overfit Gap by Depth")
axes[1].set_ylabel("Train AUC - Val AUC")
axes[1].axhline(y=0.05, color="gray", linestyle="--", alpha=0.5, label="0.05 threshold")
axes[1].legend()
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("rf_depth_analysis.png", dpi=150, bbox_inches="tight")
mlflow.log_artifact("rf_depth_analysis.png")
plt.show()
print("Saved and logged to MLflow")